# Cell Dataset Creation

Will download and prepare the cell dataset for training

## Download and extract dataset

In [11]:
import requests
import pathlib
import albumentations as alt
import zipfile

data_path = pathlib.Path('./data/')
data_path.mkdir(exist_ok=True)
zip_path = data_path / "pbc_dataset.zip"
extract_path = data_path / "pbc_dataset"
extract_path.mkdir(exist_ok=True)
pbc_base_path = extract_path / "dataset"

In [8]:
URLS = {
    'pbc_dataset.zip': 'https://zenodo.org/records/17333317/files/dataset.zip?download=1',
    'pbc_meta.csv': "https://zenodo.org/records/17333317/files/metadata.csv?download=1",
}
for filename, url in URLS.items():
    response = requests.get(url)
    if response.status_code == 200:
        with open(data_path / filename, 'wb') as f:
            f.write(response.content)
    else:
        print(response)

In [3]:

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

NameError: name 'data_path' is not defined

In [6]:
import pandas as pd

df = pd.read_csv(data_path / 'pbc_meta.csv')
df["split"] = df['path'].apply(lambda x: x.split('/')[0])
df["label"] = df['path'].apply(lambda x: x.split('/')[-1])
df['path'] = df['path'] + '/' + df['image_name']
df = df.drop(columns=['patient_id', 'image_name'])

df.head()

NameError: name 'extract_path' is not defined

In [32]:
sorted_labels = sorted(df['label'].unique().tolist())
label_id_map = {l:i for i, l in enumerate(sorted_labels)}
df['label'] = df['label'].apply(lambda x: label_id_map[x])

In [34]:
df.tail()

,path,split,label
31479,test/segmented_neutrophil/1604394691_117.jpg,test,12
31480,test/segmented_neutrophil/1604394691_121.jpg,test,12
31481,test/segmented_neutrophil/1604394691_138.jpg,test,12
31482,test/segmented_neutrophil/1604413745_019.jpg,test,12
31483,test/segmented_neutrophil/1604413745_061.jpg,test,12


In [17]:
import ray
import numpy as np
from ray.data.datasource.partitioning import Partitioning

ray.init()

root = pbc_base_path / "train"
partitioning = Partitioning("dir", field_names=["class"], base_dir=root)
train_ds = ray.data.read_images(root, size=(368, 368), partitioning=partitioning)
train_ds.schema()

2026-05-05 21:10:21,051	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
2026-05-05 21:10:24,137	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-05 21:10:24,149	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\andre\AppData\Local\Temp\ray\session_2026-05-05_21-10-15_381464_36596\logs\ray-data
2026-05-05 21:10:24,150	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadImage] -> LimitOperator[limit=1]
2026-05-05 21:10:24,936	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_1_0 =======
2026-05-05 21:10:24,937	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-05 21:10:24,938	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/2.4GiB object store
2026-05-05 21:10:24,938	INFO logging_progress.py:181 -- 
2026-05-05 21:10:24,939	INFO logging_progr

Column  Type
------  ----
image   ArrowTensorTypeV2(shape=(368, 368, 3), dtype=uint8)
class   string

(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py", line 628 in job_logging_config
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py", line 2801 in disconnect
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py", line 2132 in shutdown
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py", line 1143 in wrapper
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\client_mode_hook.py", line 107 in wrapper
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py", line 628 in Windows fatal exception: access violation
(raylet) 
(raylet) job_logging_config
(raylet)   File "C:\Users\andre\Projects\the-haem-dream\.venv\lib\site-packages\ray\_private\worker.py"Stack 

In [18]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import densenet121
import ray.train.torch
from ray.train import ScalingConfig, RunConfig
from ray.train import Checkpoint
from ray.train.torch import TorchTrainer

def train_func(config):  
    data_shard = ray.train.get_dataset_shard("train")
    
    transformations = transforms.Compose[
        transforms.ToTensor(),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
    ]

    data_shard.map_batches(transformations)
    
    model = densenet121(weights=None)
    model.classifier = nn.Linear(config["input_features"], config["num_classes"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    model = ray.train.torch.prepare_model(model)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    
    model.train()
    for epoch in range(config["epochs"]):
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        checkpoint_dir = "checkpoints"
    torch.save(model.state_dict(), f"{checkpoint_dir}/model.pth")
    checkpoint = Checkpoint.from_directory(checkpoint_dir)
    train.report(metrics={"loss": loss}, checkpoint=checkpoint)

run_config_path = str(pathlib.Path("run_config/").absolute())
trainer = TorchTrainer(
    train_func,
    datasets={"train": train_ds.limit(50)},
    scaling_config=ScalingConfig(num_workers=1, use_gpu=False),
    run_config=RunConfig(storage_path=run_config_path),
    train_loop_config={"lr": 1e-3, "input_features": (368*368*3), "num_classes": 13, "epochs": 200},
)
result = trainer.fit()

(TrainController pid=31952) Exception raised in creation task: The actor died because of an error raised in its creation task, ray::TrainController.__init__() (pid=31952, ip=127.0.0.1, actor_id=b4d5b76bd3557f2198899b9b01000000, repr=<ray.train.v2._internal.execution.controller.controller.TrainController object at 0x0000026F0313F970>)
(TrainController pid=31952)   File "python\\ray\\_raylet.pyx", line 1857, in ray._raylet.execute_task
(TrainController pid=31952)   File "python\\ray\\_raylet.pyx", line 1794, in ray._raylet.execute_task.function_executor
(TrainController pid=31952)   File "python\\ray\\_raylet.pyx", line 4524, in ray._raylet.CoreWorker.run_async_func_or_coro_in_event_loop
(TrainController pid=31952)   File "C:\Users\andre\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\lib\concurrent\futures\_base.py", line 451, in result
(TrainController pid=31952)     return self.__get_result()
(TrainController pid=31952)   File "C:\Users\andre\AppData\Roaming\uv\python\cpyth

ActorDiedError: The actor died because of an error raised in its creation task, [36mray::TrainController.__init__()[39m (pid=31952, ip=127.0.0.1, actor_id=b4d5b76bd3557f2198899b9b01000000, repr=<ray.train.v2._internal.execution.controller.controller.TrainController object at 0x0000026F0313F970>)
  File "python\\ray\\_raylet.pyx", line 1857, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 1794, in ray._raylet.execute_task.function_executor
  File "python\\ray\\_raylet.pyx", line 4524, in ray._raylet.CoreWorker.run_async_func_or_coro_in_event_loop
  File "C:\Users\andre\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\lib\concurrent\futures\_base.py", line 451, in result
    return self.__get_result()
  File "C:\Users\andre\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\lib\concurrent\futures\_base.py", line 403, in __get_result
    raise self._exception
  File "python\\ray\\_raylet.pyx", line 4511, in async_func
  File "C:\Users\andre\Projects

In [16]:
ray.shutdown()